<a href="https://colab.research.google.com/github/vamsiram89/ML-Projects/blob/ML/Student_Placement_KNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Student Placement Prediction with KNN

This notebook:
1. Loads the dataset from CSV  
2. Converts categorical columns to numerical (one-hot encoding)  
3. Splits features (**X**) and target (**y = placement_status**)  
4. Performs train-test split  
5. Trains a **K-Nearest Neighbors (KNN)** classifier  
6. Evaluates the model and shows results + sample predictions  


In [13]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [14]:
# Load dataset
df = pd.read_csv("/content/sample_data/Student_Placement_Career_Prediction_Dataset.csv")
df.head()


,student_id,age,gender,branch,cgpa,internship_count,project_count,certifications_count,coding_skills_score,communication_skills_score,soft_skills_score,hackathon_participation,placement_status
0,1,23,Male,IT,8.99,4,6,2,41,71,59,Yes,Placed
1,2,24,Female,Mechanical,8.36,4,7,5,42,64,97,No,Placed
2,3,22,Female,EEE,7.18,4,5,2,84,98,51,No,Placed
3,4,24,Female,Mechanical,6.28,0,2,4,60,100,92,No,Not Placed
4,5,24,Female,IT,5.30,3,4,1,78,61,77,No,Not Placed


In [15]:
# Quick inspection
display(df.info())
print("\nTarget value counts:")
print(df['placement_status'].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   student_id                  10000 non-null  int64  
 1   age                         10000 non-null  int64  
 2   gender                      10000 non-null  object 
 3   branch                      10000 non-null  object 
 4   cgpa                        10000 non-null  float64
 5   internship_count            10000 non-null  int64  
 6   project_count               10000 non-null  int64  
 7   certifications_count        10000 non-null  int64  
 8   coding_skills_score         10000 non-null  int64  
 9   communication_skills_score  10000 non-null  int64  
 10  soft_skills_score           10000 non-null  int64  
 11  hackathon_participation     10000 non-null  object 
 12  placement_status            10000 non-null  object 
dtypes: float64(1), int64(8), object(

None


Target value counts:
placement_status
Placed        5074
Not Placed    4926
Name: count, dtype: int64


In [16]:
df.shape

(10000, 13)

In [17]:
# -----------------------------
# Preprocessing
# -----------------------------

# Split X and y
y = df["placement_status"]
X = df.drop(columns=["placement_status"])

# Drop ID column if present (not useful for learning patterns)
if "student_id" in X.columns:
    X = X.drop(columns=["student_id"])

# One-hot encode categorical columns
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
print("Categorical columns:", cat_cols)

X_enc = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Encode target labels (Placed / Not Placed) -> (1/0)
le = LabelEncoder()
y_enc = le.fit_transform(y)

print("\nEncoded feature shape:", X_enc.shape)
print("Target classes:", list(le.classes_))

Categorical columns: ['gender', 'branch', 'hackathon_participation']

Encoded feature shape: (10000, 15)
Target classes: ['Not Placed', 'Placed']


In [18]:
X_enc.head()

,age,cgpa,internship_count,project_count,certifications_count,coding_skills_score,communication_skills_score,soft_skills_score,gender_Male,branch_Civil,branch_ECE,branch_EEE,branch_IT,branch_Mechanical,hackathon_participation_Yes
0,23,8.99,4,6,2,41,71,59,True,False,False,False,True,False,True
1,24,8.36,4,7,5,42,64,97,False,False,False,False,False,True,False
2,22,7.18,4,5,2,84,98,51,False,False,False,True,False,False,False
3,24,6.28,0,2,4,60,100,92,False,False,False,False,False,True,False
4,24,5.30,3,4,1,78,61,77,False,False,False,False,True,False,False


In [19]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_enc, y_enc,
    test_size=0.2,
    random_state=42,)

print("Train:", X_train.shape, " Test:", X_test.shape)


Train: (8000, 15)  Test: (2000, 15)


In [20]:
# Feature scaling (important for KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [21]:
# -----------------------------
# KNN Model Training
# -----------------------------

# You can tune n_neighbors; starting with 5 is common
knn = KNeighborsClassifier(n_neighbors=15) # k = 5
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)

In [22]:
# -----------------------------
# Evaluation
# -----------------------------
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=le.classes_)

print("Accuracy:", round(acc, 4))
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", report)

Accuracy: 0.906

Confusion Matrix:
 [[851 100]
 [ 88 961]]

Classification Report:
               precision    recall  f1-score   support

  Not Placed       0.91      0.89      0.90       951
      Placed       0.91      0.92      0.91      1049

    accuracy                           0.91      2000
   macro avg       0.91      0.91      0.91      2000
weighted avg       0.91      0.91      0.91      2000



In [23]:
# Sample predictions (first 15 rows of test set)
sample_df = X_test.copy()
sample_df["Actual"] = le.inverse_transform(y_test)
sample_df["Predicted"] = le.inverse_transform(y_pred)

display(sample_df.head(15)[["Actual", "Predicted"]])

,Actual,Predicted
6252,Not Placed,Not Placed
4684,Not Placed,Not Placed
1731,Placed,Placed
4742,Not Placed,Not Placed
4521,Not Placed,Not Placed
6340,Placed,Placed
576,Not Placed,Placed
5202,Not Placed,Not Placed
6363,Placed,Placed
439,Placed,Placed


# (Optional) Quick tuning for best k in a small range
# Uncomment to run if you want to try different k values.


In [24]:
ks = range(1, 21)
scores = []
for k in ks:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    scores.append(accuracy_score(y_test, pred))

best_k = ks[int(np.argmax(scores))]
print("Best k:", best_k, "Accuracy:", max(scores))

Best k: 15 Accuracy: 0.906
